# Carga de Documentos y Embeddings a Azure AI Search (RAG)
Este notebook ingiere documentos, genera embeddings con **Azure OpenAI** y los indexa en **Azure AI Search** (Vector Search).

In [ ]:
# !pip install azure-search-documents openai azure-identity tiktoken unstructured
import os, json, glob
from typing import List
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import SearchIndex, SimpleField, SearchFieldDataType, VectorSearch, VectorSearchProfile, HnswAlgorithmConfiguration
from azure.core.credentials import AzureKeyCredential
from openai import AzureOpenAI

# === Configuración ===
AOAI_ENDPOINT = os.getenv('AOAI_ENDPOINT', 'https://<tu-aoai>.openai.azure.com/')
AOAI_API_KEY = os.getenv('AOAI_KEY', '<TU_API_KEY>')
AOAI_EMBED_DEPLOY = os.getenv('AOAI_EMBED_DEPLOY', 'text-embedding-3-small')
SEARCH_ENDPOINT = os.getenv('SEARCH_ENDPOINT', 'https://llm-search-001.search.windows.net')
SEARCH_API_KEY = os.getenv('SEARCH_KEY', '<TU_SEARCH_KEY>')
SEARCH_INDEX = os.getenv('SEARCH_INDEX', 'docs-index')
EMBED_DIM = int(os.getenv('EMBED_DIM', '1536'))
DATA_DIR = os.getenv('DATA_DIR', './data')

client = AzureOpenAI(api_key=AOAI_API_KEY, api_version='2024-06-01', azure_endpoint=AOAI_ENDPOINT)
search_cred = AzureKeyCredential(SEARCH_API_KEY)
search_client = SearchClient(endpoint=SEARCH_ENDPOINT, index_name=SEARCH_INDEX, credential=search_cred)
index_client = SearchIndexClient(endpoint=SEARCH_ENDPOINT, credential=search_cred)

print('Config OK:')
for k in ['AOAI_ENDPOINT','AOAI_EMBED_DEPLOY','SEARCH_ENDPOINT','SEARCH_INDEX','EMBED_DIM','DATA_DIR']:
    print(f'- {k} =', eval(k))

In [ ]:
def ensure_index(index_name: str, dim: int):
    # Crea el índice si no existe
    try:
        idx = index_client.get_index(index_name)
        print('Index exists:', idx.name)
        return
    except Exception:
        pass

    fields = [
        SimpleField(name='id', type=SearchFieldDataType.String, key=True, filterable=True),
        SimpleField(name='content', type=SearchFieldDataType.String, searchable=True),
        SimpleField(name='metadata', type=SearchFieldDataType.String, searchable=True, filterable=True),
        # Nota: para vector field, usamos el esquema JSON cuando se crea vía REST.
    ]

    vector_search = VectorSearch(
        algorithms=[HnswAlgorithmConfiguration(name='hnsw-config')],
        profiles=[VectorSearchProfile(name='vector-profile', algorithm_configuration_name='hnsw-config')]
    )

    # Crear índice base (sin vector vía SDK simplificado). Para vector exacto, usar REST/ARM template.
    index = SearchIndex(name=index_name, fields=fields, vector_search=vector_search)
    index_client.create_index(index)
    print('Index created (base). Asegúrate de añadir el campo vectorial vía REST si no existía).')

ensure_index(SEARCH_INDEX, EMBED_DIM)

In [ ]:
def embed(text: str) -> List[float]:
    # Genera embeddings usando tu deployment de embeddings
    r = client.embeddings.create(model=AOAI_EMBED_DEPLOY, input=text)
    return r.data[0].embedding

def load_docs(data_dir: str) -> List[dict]:
    docs = []
    for path in glob.glob(f'{data_dir}/*.txt'):
        with open(path, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()
            docs.append({'id': os.path.basename(path), 'content': content, 'metadata': f'file={os.path.basename(path)}'})
    if not docs:
        # crear demo
        os.makedirs(data_dir, exist_ok=True)
        demo = os.path.join(data_dir, 'demo.txt')
        with open(demo, 'w', encoding='utf-8') as f:
            f.write('Azure OpenAI + Azure AI Search demo document for RAG indexing.')
        docs.append({'id':'demo.txt','content':'Azure OpenAI + Azure AI Search demo document for RAG indexing.','metadata':'file=demo.txt'})
    return docs

docs = load_docs(DATA_DIR)
len(docs), docs[0]['id']

In [ ]:
def upsert_documents(docs: List[dict]):
    batch = []
    for d in docs:
        vec = embed(d['content'])
        # Documento extendido: incluye vector en campo 'contentVector'
        item = {
            'id': d['id'],
            'content': d['content'],
            'metadata': d['metadata'],
            'contentVector': vec
        }
        batch.append({'@search.action':'mergeOrUpload', **item})
    r = search_client.upload_documents(documents=batch)
    print('Indexed:', r)

upsert_documents(docs)

In [ ]:
# Consulta vectorial de ejemplo:
query = '¿Qué es Azure OpenAI y cómo se integra con Azure AI Search?'
q_vec = embed(query)
results = search_client.search(search_text=None, vector={'value': q_vec, 'k': 3, 'fields': 'contentVector'})
for r in results:
    print(r['id'], '->', r['content'][:80], '...')

## Opción (producción): DefaultAzureCredential y Key Vault
Para producción, usa **DefaultAzureCredential** y almacena las llaves en **Azure Key Vault**; evita exponer API keys.

In [ ]:
# from azure.identity import DefaultAzureCredential
# cred = DefaultAzureCredential()
# # Si tu Search usa RBAC en lugar de API key, puedes usar cred en SearchClient:
# # search_client = SearchClient(endpoint=SEARCH_ENDPOINT, index_name=SEARCH_INDEX, credential=cred)
# # Y para AOAI, usa Managed Identity con el SDK cuando esté soportado para tu flujo.


## Chunking con tiktoken (mejora de calidad RAG)
Dividir documentos en fragmentos solapados mejora la recuperación y la relevancia de las respuestas.

In [ ]:
# !pip install tiktoken
import tiktoken
def chunk_text(text: str, chunk_size_tokens=400, chunk_overlap_tokens=60, encoding_name='cl100k_base'):
    enc = tiktoken.get_encoding(encoding_name)
    tokens = enc.encode(text)
    chunks = []
    i = 0
    while i < len(tokens):
        window = tokens[i:i+chunk_size_tokens]
        chunk = enc.decode(window)
        chunks.append(chunk)
        i += (chunk_size_tokens - chunk_overlap_tokens)
    return chunks


In [ ]:
def upsert_documents_chunked(docs):
    for d in docs:
        parts = chunk_text(d['content'], 400, 60)
        batch = []
        for idx, part in enumerate(parts):
            vec = embed(part)
            item = {
                'id': f"{d['id']}#{idx:04d}",
                'content': part,
                'metadata': f"{d.get('metadata','')}|chunk={idx}",
                'contentVector': vec
            }
            batch.append({'@search.action':'mergeOrUpload', **item})
        print('Uploading', len(batch), 'chunks for', d['id'])
        r = search_client.upload_documents(documents=batch)
        print('Result:', r)

# Ejecuta la versión con chunking
upsert_documents_chunked(docs)
